# Optimal Threshold Selection (FAR vs FRR)

This notebook analyzes genuine vs. impostor face matching similarity scores, calculates False Accept Rate (FAR) and False Reject Rate (FRR) curves, and determines the optimal threshold based on Equal Error Rate (EER).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from src.evaluation.metrics import FaceRecognitionEvaluator
from src.evaluation.visualizations import plot_far_frr, plot_score_distributions

print('Libraries and dependencies loaded.')

### Step 1: Generate or Load Verification Similarity Scores
We load or simulate genuine matching scores (similarities from same identities) and impostor matching scores (similarities from different identities).

In [ ]:
# Simulated Cosine Similarity Scores in the [0, 1] range
np.random.seed(42)
genuine_scores = np.random.normal(loc=0.82, scale=0.08, size=5000)
impostor_scores = np.random.normal(loc=0.35, scale=0.12, size=5000)

# Clip values to valid cosine similarity ranges
genuine_scores = np.clip(genuine_scores, 0, 1)
impostor_scores = np.clip(impostor_scores, 0, 1)

print(f"Loaded {len(genuine_scores)} genuine scores and {len(impostor_scores)} impostor scores.")

### Step 2: Compute FAR and FRR over a range of thresholds
We compute error rates for thresholds starting from 0.0 to 1.0 to find the point where FAR equals FRR.

In [ ]:
thresholds = np.linspace(0.0, 1.0, 100)
far, frr = FaceRecognitionEvaluator.compute_far_frr(genuine_scores, impostor_scores, thresholds)
eer, optimal_threshold = FaceRecognitionEvaluator.compute_eer(genuine_scores, impostor_scores)

print(f"Equal Error Rate (EER): {eer*100:.3f}%")
print(f"Optimal Decision Threshold (EER point): {optimal_threshold:.3f}")

### Step 3: Visualize Score Distributions
Visualizing the score distributions helps us see the overlap between genuine and impostor matches.

In [ ]:
plt.figure(figsize=(10, 6))
sns.histplot(genuine_scores, color='green', label='Genuine (Same Identity)', kde=True, stat="density", bins=50, alpha=0.5)
sns.histplot(impostor_scores, color='red', label='Impostor (Different Identity)', kde=True, stat="density", bins=50, alpha=0.5)
plt.axvline(x=optimal_threshold, color='blue', linestyle='--', label=f'Decision Threshold ({optimal_threshold:.3f})')
plt.xlabel('Cosine Similarity Score')
plt.ylabel('Density')
plt.title('Genuine vs Impostor Similarity Score Distributions')
plt.legend()
plt.show()

### Step 4: Plot FAR vs FRR Curves
This plot shows FAR and FRR error rates corresponding to each decision threshold.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(thresholds, far * 100, label='False Accept Rate (FAR)', color='red', lw=2)
plt.plot(thresholds, frr * 100, label='False Reject Rate (FRR)', color='blue', lw=2)
plt.axvline(x=optimal_threshold, color='gray', linestyle='--', label=f'EER Threshold ({optimal_threshold:.3f})')
plt.scatter([optimal_threshold], [eer * 100], color='black', zorder=5, label=f'EER = {eer*100:.2f}%')
plt.xlabel('Decision Threshold')
plt.ylabel('Error Rate (%)')
plt.title('False Accept Rate (FAR) vs False Reject Rate (FRR)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()